### Initialization

In [0]:
%pip install llama-index-core
dbutils.library.restartPython()

In [0]:
import pyspark.sql.functions as F
import pandas as pd
from llama_index.core.node_parser import SentenceSplitter

### Chunking

In [0]:
CATALOG = "humanoid_robotics"
SILVER  = f"{CATALOG}.silver"
TARGET_TABLE = f"{SILVER}.doc_chunks"

CHUNKING_VERSION = "v1"   # bump when chunk size, overlap or text layout changes
CHUNK_SIZE    = 512       # tokens
CHUNK_OVERLAP = 64        # tokens

### Load: Read Silver Tables

In [0]:
df_docs = spark.sql(f"""
    WITH doc_companies AS (
        -- companies found in extracted events (already resolved to company_id)
        SELECT doc_id, company_id
        FROM {SILVER}.event_companies
        WHERE company_id IS NOT NULL

        UNION   -- removes duplicates

        -- the primary company recorded in the manifest
        SELECT d.doc_id, co.company_id
        FROM {SILVER}.documents d
        JOIN {SILVER}.company co
          ON upper(trim(co.company_name)) = upper(trim(d.primary_company))
    ),

    company_list AS (
        -- one row per document, companies as a space-separated string
        SELECT doc_id,
               array_join(sort_array(collect_set(company_id)), ' ') AS company_ids
        FROM doc_companies
        GROUP BY doc_id
    )

    SELECT c.doc_id,
           c.clean_text,
           d.title,
           d.url,
           d.publisher,
           d.source_type,
           d.language,
           d.publish_date,
           year(d.publish_date)         AS publish_year,
           coalesce(cl.company_ids, '') AS company_ids
    FROM {SILVER}.documents_clean c
    JOIN {SILVER}.documents d
      ON d.doc_id = c.doc_id
    JOIN {SILVER}.documents_relevance r
      ON r.doc_id = c.doc_id
     AND r.is_relevant                  -- only keep relevant documents
    LEFT JOIN company_list cl
      ON cl.doc_id = c.doc_id
""")

display(df_docs)

In [0]:
n_relevant = spark.table(f"{SILVER}.documents_relevance").filter("is_relevant").count()
print(f"relevant documents: {n_relevant}")
print(f"loaded documents:   {df_docs.count()}")   # expect equal

In [0]:
df_docs.display()

### Chunking

In [0]:
splitter = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)


def split_document(doc: dict) -> list[dict]:
    """Split one document into chunks. Metadata is copied onto every chunk
    because the vector index cannot join back to other tables."""
    chunks = splitter.split_text(doc["clean_text"])

    return [
        {
            "chunk_id":       f"{doc['doc_id']}_C{i:02d}",
            "doc_id":         doc["doc_id"],
            "chunk_index":    i,
            "chunk_text":     chunk,
            "embedding_text": f"{doc['title']}\n\n{chunk}",
            "title":          doc["title"],
            "url":            doc["url"],
            "publisher":      doc["publisher"],
            "source_type":    doc["source_type"],
            "language":       doc["language"],
            "publish_date":   doc["publish_date"],
            "publish_year":   doc["publish_year"],
            "company_ids":    doc["company_ids"],
        }
        for i, chunk in enumerate(chunks)
    ]

In [0]:
# Small corpus: collect to the driver and chunk in plain Python.
# For a larger corpus, the same split_document() would run on the executors via mapInPandas.
docs = [row.asDict() for row in df_docs.collect()]

chunk_rows = []
for doc in docs:
    chunk_rows.extend(split_document(doc))

print(f"{len(docs)} documents -> {len(chunk_rows)} chunks")

In [0]:
CHUNK_SCHEMA = """
    chunk_id STRING, doc_id STRING, chunk_index INT,
    chunk_text STRING, embedding_text STRING,
    title STRING, url STRING, publisher STRING, source_type STRING,
    language STRING, publish_date DATE, publish_year INT, company_ids STRING
"""

df_chunks = (
    spark.createDataFrame(chunk_rows, schema=CHUNK_SCHEMA)
    .withColumn("chunking_version", F.lit(CHUNKING_VERSION))
    .withColumn("chunked_time", F.current_timestamp())
)

display(df_chunks)

### Write into Silver Table

In [0]:
(df_chunks.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE))

# Change Data Feed is required by the Delta Sync vector index
spark.sql(f"ALTER TABLE {TARGET_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

### Data Inspection

In [0]:
df = spark.table(TARGET_TABLE)

total_chunks = df.count()
dup_ids      = total_chunks - df.select("chunk_id").distinct().count()
empty_chunks = df.filter(F.length(F.trim("chunk_text")) == 0).count()
docs_out     = df.select("doc_id").distinct().count()

print(f"chunks:          {total_chunks}")
print(f"duplicate ids:   {dup_ids}")        # expect 0
print(f"empty chunks:    {empty_chunks}")   # expect 0
print(f"documents out:   {docs_out}")       # expect = loaded documents

display(
    df.groupBy("doc_id")
      .agg(F.count("*").alias("n_chunks"),
           F.min(F.length("chunk_text")).alias("min_chars"),
           F.max(F.length("chunk_text")).alias("max_chars"))
      .orderBy("doc_id")
)

In [0]:
%sql
SELECT chunk_id, chunk_index, length(chunk_text) AS chars, left(chunk_text, 150) AS preview
FROM humanoid_robotics.silver.doc_chunks
ORDER BY chunk_id;